# BrainMaxxing on Colab

Runs the full pipeline (download → preprocess → spectra → parameterise → stats → figures) for
ds003969 with **~1 GB of disk at any time**. Outputs (~4.5 MB/subject) are checkpointed to Google
Drive so the run survives Colab's session limits: re-run the notebook and it resumes.

Colab free tier has 2 vCPUs; ICA is the bottleneck at ~4 min per block, so budget
**~15 min per subject, ~25 h for all 98** across several sessions. Start with `N_SUBJECTS = 10`.

Runtime → Change runtime type → CPU is fine (no GPU is used).

In [ ]:
#@title 1. Get the code and install dependencies
REPO_URL = "https://github.com/cation21/Gamma.git"  #@param {type:"string"}
import os, subprocess, sys
if not os.path.exists("/content/BrainMaxxing"):
    subprocess.run(["git", "clone", REPO_URL, "/content/BrainMaxxing"], check=True)
%cd /content/BrainMaxxing
!git pull -q
!pip install -q -e . 2>&1 | tail -1
print("ok")

In [ ]:
#@title 2. Mount Google Drive and write the Colab config
from google.colab import drive
drive.mount("/content/drive")

import yaml, pathlib
cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["dataset"]["bids_root"] = "/content/ds003969"                              # ephemeral: raw is deleted per subject
cfg["dataset"]["derivatives"] = "/content/drive/MyDrive/BrainMaxxing/derivatives"  # persistent checkpoints
cfg["dataset"]["tables"] = "/content/drive/MyDrive/BrainMaxxing/tables"
cfg["dataset"]["qc"] = "/content/drive/MyDrive/BrainMaxxing/qc"
pathlib.Path("configs/colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(open("configs/colab.yaml").read()[:400])

In [ ]:
#@title 3. Stream subjects: download -> process -> delete raw (resumable)
N_SUBJECTS = 10  #@param {type:"integer"}
N_JOBS = 2       #@param {type:"integer"}
!python scripts/07_stream.py --config configs/colab.yaml --n-first $N_SUBJECTS --n-jobs $N_JOBS 2>&1 | grep -E "07_stream|FAILED|Traceback" 

In [ ]:
#@title 4. Parameterise, model, plot
!python scripts/04_parameterize.py --config configs/colab.yaml --multiverse 2>&1 | grep -E "wrote|Error"
!python scripts/05_stats.py --config configs/colab.yaml 2>&1 | grep -vE "Warning|warn"
!python scripts/06_figures.py --config configs/colab.yaml 2>&1 | grep -E "INFO|Error"
from IPython.display import Image, display
for f in ["group_spectra", "interaction_exponent", "interaction_naive_gamma_power", "interaction_periodic_gamma_power"]:
    display(Image(f"figures/{f}.png"))

## Notes
- Re-running cell 3 skips subjects whose outputs already exist on Drive.
- `--n-jobs 2` matches the free tier; on Colab Pro try 4.
- Results tables land in `MyDrive/BrainMaxxing/tables/`; copy them into the repo's `results/tables/` to commit.